# Experiment 4: Term Frequency Analysis, Named Entity Recognition, and TF-IDF

This notebook demonstrates vocabulary frequency analysis using spaCy and pure Python, Named Entity Recognition (NER), and a step-by-step multi-document TF-IDF mathematical pipeline built from scratch.

## 1. Setup and Environment Imports

Import required standard libraries (`string`, `math`), data structures (`collections.Counter`), formatting tool (`pandas`), and the spaCy English pipeline.

In [1]:
import string
import math
import pandas as pd
from collections import Counter
import spacy

nlp = spacy.load('en_core_web_sm')

## Experiment 4.1: Term-Frequency Analysis and Named Entity Recognition Using an NLP Toolkit

Perform word frequency counting using spaCy tokenization, export results to CSV, display the top 10 terms, and extract named entities.

In [2]:
with open('4.1_4.2_input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

doc = nlp(text)
words = [token.text.lower() for token in doc if token.is_alpha]
tf = Counter(words)

df_tf_toolkit = pd.DataFrame(tf.most_common(), columns=['Term', 'Frequency'])
df_tf_toolkit.to_csv('4.1_term_frequency_toolkit.csv', index=False)

print("Top 10 Most Frequent Terms (Toolkit):")
print(df_tf_toolkit.head(10))

print("\nSample Named Entities:")
seen_entities = set()
for ent in doc.ents:
    if ent.text not in seen_entities:
        print(f"{ent.text:25} -> {ent.label_:10} ({spacy.explain(ent.label_)})")
        seen_entities.add(ent.text)
    if len(seen_entities) >= 10:
        break

Top 10 Most Frequent Terms (Toolkit):
   Term  Frequency
0     a         88
1   and         86
2   the         75
3    of         42
4    to         42
5    in         42
6   can         42
7    is         37
8   may         30
9  word         30

Sample Named Entities:
Natural Language Processing and Artificial Intelligence

Natural Language Processing -> ORG        (Companies, agencies, institutions, etc.)
NLP                       -> ORG        (Companies, agencies, institutions, etc.)
English                   -> LANGUAGE   (Any named language)
Hindi                     -> GPE        (Countries, cities, states)
Assamese                  -> NORP       (Nationalities or religious or political groups)
Bengali                   -> NORP       (Nationalities or religious or political groups)
Tamil                     -> GPE        (Countries, cities, states)
twenty                    -> CARDINAL   (Numerals that do not fall under another type)
Python                    -> GPE        (Cou

## Experiment 4.2: Term-Frequency Analysis Without Using Any NLP Toolkit (Pure Python)

Calculate word frequencies using standard Python dictionaries, export results to CSV, and identify the top 10 most frequent terms.

In [3]:
with open('4.1_4.2_input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

clean_text = text.lower().translate(str.maketrans('', '', string.punctuation))
words = clean_text.split()

tf_dict = {}
for word in words:
    tf_dict[word] = tf_dict.get(word, 0) + 1

sorted_tf = sorted(tf_dict.items(), key=lambda x: x[1], reverse=True)
df_tf_pure = pd.DataFrame(sorted_tf, columns=['Term', 'Frequency'])
df_tf_pure.to_csv('4.2_term_frequency_pure_python.csv', index=False)

print("Top 10 Most Frequent Terms (Pure Python):")
print(df_tf_pure.head(10))

Top 10 Most Frequent Terms (Pure Python):
   Term  Frequency
0     a         90
1   and         86
2   the         75
3    to         42
4    in         42
5   can         41
6    of         40
7    is         37
8   may         30
9  word         30


## Experiment 4.3: Manual Multi-Document TF-IDF Calculation From Scratch

Step-by-step calculation of Term Frequency (TF), Document Frequency (DF), Inverse Document Frequency (IDF), and TF-IDF scores across three documents without using any external NLP library.

In [4]:
documents = [
    "Natural language processing is a field of artificial intelligence.",
    "Natural language processing helps computers understand human language.",
    "Machine learning is an important part of artificial intelligence."
]

cleaned_docs = []
for doc in documents:
    clean_text = doc.lower().translate(str.maketrans('', '', string.punctuation))
    tokens = clean_text.split()
    cleaned_docs.append(tokens)

vocab = sorted(list(set(word for doc in cleaned_docs for word in doc)))
N = len(documents)

df_dict = {}
for word in vocab:
    count = 0
    for doc in cleaned_docs:
        if word in doc:
            count += 1
    df_dict[word] = count

idf_dict = {}
for word in vocab:
    idf_dict[word] = math.log(N / df_dict[word])

df_idf_table = pd.DataFrame({
    'Term': vocab,
    'DF': [df_dict[w] for w in vocab],
    'IDF': [round(idf_dict[w], 4) for w in vocab]
})

print("=== 1. Vocabulary, Document Frequency (DF) & Inverse Document Frequency (IDF) ===")
print(df_idf_table)

for i, doc in enumerate(cleaned_docs, 1):
    doc_len = len(doc)
    doc_scores = []
    
    for word in vocab:
        word_count = doc.count(word)
        if word_count > 0:
            tf = word_count / doc_len
            idf = idf_dict[word]
            tfidf = tf * idf
            doc_scores.append({
                'Term': word,
                'Count': word_count,
                'TF': round(tf, 4),
                'DF': df_dict[word],
                'IDF': round(idf, 4),
                'TF-IDF': round(tfidf, 4)
            })
            
    df_doc_table = pd.DataFrame(doc_scores)
    print(f"\n=== Document {i} TF & TF-IDF Scores ===")
    print(df_doc_table)
    
    top_terms = df_doc_table.sort_values(by='TF-IDF', ascending=False)
    print(f"\nTop Terms for Document {i}:")
    print(top_terms[['Term', 'TF-IDF']].head(3))

=== 1. Vocabulary, Document Frequency (DF) & Inverse Document Frequency (IDF) ===
            Term  DF     IDF
0              a   1  1.0986
1             an   1  1.0986
2     artificial   2  0.4055
3      computers   1  1.0986
4          field   1  1.0986
5          helps   1  1.0986
6          human   1  1.0986
7      important   1  1.0986
8   intelligence   2  0.4055
9             is   2  0.4055
10      language   2  0.4055
11      learning   1  1.0986
12       machine   1  1.0986
13       natural   2  0.4055
14            of   2  0.4055
15          part   1  1.0986
16    processing   2  0.4055
17    understand   1  1.0986

=== Document 1 TF & TF-IDF Scores ===
           Term  Count      TF  DF     IDF  TF-IDF
0             a      1  0.1111   1  1.0986  0.1221
1    artificial      1  0.1111   2  0.4055  0.0451
2         field      1  0.1111   1  1.0986  0.1221
3  intelligence      1  0.1111   2  0.4055  0.0451
4            is      1  0.1111   2  0.4055  0.0451
5      language      1